# Programación orientada a objetos con Python
## Video 2. Clases e instancias: `class`, `__init__`, `self`, atributos de instancia y de clase

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Explicar qué ocurre paso a paso cuando se define una clase con `class` y se crea una instancia llamando a la clase.
2. Describir el papel del método inicializador `__init__` y del primer parámetro `self`.
3. Distinguir entre atributos de instancia (estado propio de cada objeto) y atributos de clase (datos compartidos por todas las instancias).
4. Evitar errores frecuentes al modificar atributos de clase o al ocultar un atributo de clase desde una instancia.

## 1. Planteamiento del problema

**Definición.** Un *vector en el plano euclidiano* $\mathbb{R}^2$ es un par ordenado $u = (x, y)$ con $x, y \in \mathbb{R}$. La *norma euclidiana* de $u = (x, y)$ es el número real no negativo

$$\|u\| = \sqrt{x^2 + y^2}.$$

Dados dos vectores $u = (x_1, y_1)$ y $v = (x_2, y_2)$ en $\mathbb{R}^2$, su *suma* es $u + v = (x_1 + x_2,\; y_1 + y_2)$ y su *producto interior usual* (o producto punto) es

$$\langle u, v \rangle = x_1 x_2 + y_1 y_2.$$

**Problema: representación y operaciones elementales en $\mathbb{R}^2$.**

- **Entrada:** pares de números reales $(x, y)$ que representan vectores del plano, junto con una tolerancia positiva $\varepsilon > 0$ compartida por todos los vectores de la clase para decidir si dos vectores son aproximadamente iguales ($\|u - v\| < \varepsilon$).
- **Salida:** instancias de una clase `VectorPlano` donde cada vector conserve sus propias coordenadas $(x, y)$ pero todas las instancias compartan la dimensión del espacio ($2$) y la tolerancia $\varepsilon$.

## 2. La sentencia `class` y el inicializador `__init__`

Al ejecutar la sentencia `class VectorPlano:`, Python crea un objeto de tipo clase y lo asocia al nombre `VectorPlano`. Dentro de su bloque se definen las funciones que formarán el comportamiento de sus objetos; cuando una función se define dentro de una clase, se denomina *método*.

Cuando escribimos `u = VectorPlano(3.0, 4.0)`, Python realiza cuatro pasos en orden:

1. Reserva en memoria un objeto nuevo vacío de tipo `VectorPlano`.
2. Llama automáticamente al método inicializador `VectorPlano.__init__(objeto_nuevo, 3.0, 4.0)`.
3. Dentro de `__init__`, el parámetro `self` hace referencia a ese objeto recién creado y permite adjuntarle sus atributos (`self.x = float(x)` y `self.y = float(y)`).
4. Devuelve la referencia al objeto ya inicializado y la asocia al nombre `u`.

Observa que `__init__` **no** debe incluir una instrucción `return` con un valor: su única tarea es inicializar el estado del objeto recién construido.

In [ ]:
from math import sqrt


class VectorPlano:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2)

    def sumar(self, otro):
        return VectorPlano(self.x + otro.x, self.y + otro.y)

    def producto_interior(self, otro):
        return self.x * otro.x + self.y * otro.y

    def mostrar(self):
        return f"({self.x}, {self.y})"


u = VectorPlano(3.0, 4.0)
v = VectorPlano(1.0, -2.0)

print("u =", u.mostrar())
print("Norma mediante instancia:", u.norma())
print("Norma mediante clase:    ", VectorPlano.norma(u))
print("Suma mediante instancia: ", u.sumar(v).mostrar())
print("Suma mediante clase:     ", VectorPlano.sumar(u, v).mostrar())
print("Producto interior:       ", u.producto_interior(v))

## 3. Atributos de instancia: estado propio de cada objeto

**Definición.** Un *atributo de instancia* es una variable asociada a un objeto particular, creada habitualmente dentro de `__init__` mediante `self.<nombre> = <valor>`.

Cada instancia conserva sus propios atributos en un diccionario interno llamado `__dict__`. Por esta razón, los datos de `u` son independientes de los datos de `v`: consultar o modificar `u.x` no altera `v.x`.

Además, las funciones incorporadas `type(objeto)` e `isinstance(objeto, Clase)` permiten inspeccionar la clase a la que pertenece una instancia.

In [ ]:
from math import gcd


u = VectorPlano(3, 4)
v = VectorPlano(-1, 2)

print("Atributos de u:", u.__dict__)
print("Atributos de v:", v.__dict__)
print("type(u) is VectorPlano:", type(u) is VectorPlano)
print("isinstance(u, VectorPlano):", isinstance(u, VectorPlano))


class Fraccion:
    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self.numerador = numerador // divisor
        self.denominador = denominador // divisor

    def sumar(self, otra):
        return Fraccion(
            self.numerador * otra.denominador + self.denominador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"


f = Fraccion(6, -8)
print("Atributos normalizados de Fraccion(6, -8):", f.__dict__)

## 4. Atributos de clase: datos compartidos por todas las instancias

**Definición.** Un *atributo de clase* es una variable definida directamente en el cuerpo de la sentencia `class`, fuera de cualquier método. Pertenece a la clase misma y es compartido por todas sus instancias.

Cuando evaluamos `objeto.atributo`, Python busca el nombre siguiendo un orden estricto:

1. Primero busca `'atributo'` en el diccionario de la instancia (`objeto.__dict__`).
2. Si no aparece allí, lo busca en el diccionario de la clase (`type(objeto).__dict__`).

Por este mecanismo, todas las instancias de `VectorPlano` pueden leer `dimension` y `tolerancia` sin almacenar una copia repetida en cada objeto.

In [ ]:
class VectorPlano:
    dimension = 2
    tolerancia = 1e-9

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2)

    def sumar(self, otro):
        return VectorPlano(self.x + otro.x, self.y + otro.y)

    def producto_interior(self, otro):
        return self.x * otro.x + self.y * otro.y

    def aproximadamente_igual(self, otro):
        diferencia = VectorPlano(self.x - otro.x, self.y - otro.y)
        return diferencia.norma() < self.tolerancia

    def mostrar(self):
        return f"({self.x}, {self.y})"


u = VectorPlano(0.1 + 0.2, 0.3)
v = VectorPlano(0.3, 0.3)

print("Desde la clase:", VectorPlano.dimension)
print("Desde u:", u.dimension, "| Desde v:", v.dimension)
print("¿'dimension' está en u.__dict__?          ", "dimension" in u.__dict__)
print("¿'dimension' está en VectorPlano.__dict__?", "dimension" in VectorPlano.__dict__)
print("u == v exactamente en sus atributos:     ", (u.x, u.y) == (v.x, v.y))
print("u aproximadamente igual a v:             ", u.aproximadamente_igual(v))

### Riesgos con atributos de clase: ocultamiento y mutabilidad

Aunque los atributos de clase son muy útiles para constantes o valores compartidos, conviene evitar dos trampas frecuentes:

1. **Ocultamiento (sombreado):** si ejecutamos `u.tolerancia = 1e-3` sobre una instancia, Python **no** modifica `VectorPlano.tolerancia`. En su lugar, crea un atributo de instancia nuevo dentro de `u.__dict__` que oculta al atributo de clase únicamente cuando se consulta `u`.
2. **Atributo de clase mutable:** si un atributo de clase almacena una lista o un diccionario y se modifica en su lugar (por ejemplo, con `.append()`), el cambio afecta de inmediato a todas las instancias pasadas y futuras porque todas apuntan al mismo objeto en memoria. Por ello, los contenedores mutables propios de cada objeto deben inicializarse siempre dentro de `__init__` sobre `self`.

In [ ]:
# Trampa 1: ocultamiento desde una instancia
u = VectorPlano(1.0, 1.0)
v = VectorPlano(1.0005, 1.0)

u.tolerancia = 1e-3  # crea un atributo de instancia solo en u

print("u.tolerancia:", u.tolerancia)
print("v.tolerancia:", v.tolerancia)
print("VectorPlano.tolerancia:", VectorPlano.tolerancia)
print("u.__dict__:", u.__dict__)
print("v.__dict__:", v.__dict__)
print("u.aproximadamente_igual(v):", u.aproximadamente_igual(v))
print("v.aproximadamente_igual(u):", v.aproximadamente_igual(u))


# Trampa 2: atributo de clase mutable compartido por error
class PolinomioMalo:
    coeficientes = []  # error: lista compartida por todas las instancias

    def __init__(self, coeficientes):
        self.coeficientes.extend(coeficientes)


p_malo = PolinomioMalo([1, 2])
q_malo = PolinomioMalo([3, 4])
print("PolinomioMalo p_malo.coeficientes:", p_malo.coeficientes)
print("PolinomioMalo q_malo.coeficientes:", q_malo.coeficientes)


# Diseño correcto: atributo de clase inmutable y copia por instancia en __init__
class PolinomioSeguro:
    variable = "x"  # seguro: las cadenas de texto son inmutables

    def __init__(self, coeficientes):
        self.coeficientes = list(coeficientes)


p_bueno = PolinomioSeguro([1, 2])
q_bueno = PolinomioSeguro([3, 4])
print("PolinomioSeguro p_bueno.coeficientes:", p_bueno.coeficientes)
print("PolinomioSeguro q_bueno.coeficientes:", q_bueno.coeficientes)

### Tabla comparativa

| Criterio | Atributo de instancia | Atributo de clase |
|---|---|---|
| Dónde se declara | Dentro de `__init__` (u otro método) con `self.nombre = valor` | Directamente en el cuerpo de `class`, fuera de los métodos |
| A quién pertenece | A una instancia concreta | A la clase (compartido por todas las instancias) |
| Diccionario donde vive | `instancia.__dict__` | `Clase.__dict__` |
| Qué modela en objetos matemáticos | Datos propios: coordenadas $(x, y)$, numerador y denominador, lista de coeficientes | Propiedades comunes: dimensión del espacio, tolerancia numérica, símbolo de la indeterminada |

## 5. Ejercicios

**Ejercicio 1 (ampliación de `Fraccion`).** Añade a la clase `Fraccion` un atributo de clase `cuerpo = "Q"` (que identifica al cuerpo de los números racionales $\mathbb{Q}$) y un método de instancia `reciproco(self)`:
- **Entrada:** una instancia `self` de `Fraccion` con numerador $p \in \mathbb{Z}$ y denominador $q \in \mathbb{Z}$ ($q > 0$).
- **Salida:** si $p \neq 0$, una nueva instancia de `Fraccion` equivalente a $(q, p)$ en forma normalizada; si $p = 0$, debe lanzar `ZeroDivisionError`.

**Ejercicio 2 (vectores en $\mathbb{R}^3$).** Define la clase `VectorEspacio` para representar vectores $u = (x, y, z) \in \mathbb{R}^3$ con atributo de clase `dimension = 3`:
- **Entrada:** tres números reales $x, y, z$ en `__init__`, y otra instancia `otro` de `VectorEspacio` que representa $v = (x', y', z')$ en el método `producto_vectorial(self, otro)`.
- **Salida:** el producto vectorial
  $$u \times v = (y z' - z y',\; z x' - x z',\; x y' - y x')$$
  como una nueva instancia de `VectorEspacio`. Incluye también un método `mostrar(self)`.

**Ejercicio 3 (primera versión de `Polinomio`).** Un polinomio con coeficientes reales en una indeterminada $x$ de grado a lo sumo $n$ tiene la forma

$$P(x) = a_0 + a_1 x + a_2 x^2 + \dots + a_n x^n.$$

Define la clase `Polinomio` con:
- Un atributo de clase inmutable `variable = "x"`.
- Un inicializador `__init__(self, coeficientes)` que reciba una secuencia no vacía de coeficientes $[a_0, a_1, \dots, a_n]$, elimine los ceros finales redundantes (conservando `[0]` para el polinomio nulo) y guarde una copia en `self.coeficientes`. Si la secuencia está vacía, debe lanzar `ValueError`.
- Un método `grado(self)` que devuelva el grado del polinomio (el índice del último coeficiente, es decir, `len(self.coeficientes) - 1`, por lo que las constantes tienen grado `0`).
- Un método `evaluar(self, valor)` que calcule $P(\text{valor})$ mediante la regla de Horner:
  - **Entrada:** un número real $\text{valor} \in \mathbb{R}$.
  - **Salida:** el número real $P(\text{valor}) = \sum_{k=0}^{n} a_k\,\text{valor}^k$.

## 6. Soluciones

In [ ]:
# Ejercicio 1
class Fraccion:
    cuerpo = "Q"

    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self.numerador = numerador // divisor
        self.denominador = denominador // divisor

    def sumar(self, otra):
        return Fraccion(
            self.numerador * otra.denominador + self.denominador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def reciproco(self):
        if self.numerador == 0:
            raise ZeroDivisionError("El cero no tiene recíproco en los números racionales.")
        return Fraccion(self.denominador, self.numerador)

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"


print("Cuerpo numérico:", Fraccion.cuerpo)
print("Recíproco de 3/5:", Fraccion(3, 5).reciproco().mostrar())
print("Recíproco de -2/7:", Fraccion(-2, 7).reciproco().mostrar())

try:
    Fraccion(0, 1).reciproco()
except ZeroDivisionError as error:
    print("Error esperado:", error)

In [ ]:
# Ejercicio 2
class VectorEspacio:
    dimension = 3

    def __init__(self, x, y, z):
        self.x = float(x)
        self.y = float(y)
        self.z = float(z)

    def producto_vectorial(self, otro):
        return VectorEspacio(
            self.y * otro.z - self.z * otro.y,
            self.z * otro.x - self.x * otro.z,
            self.x * otro.y - self.y * otro.x,
        )

    def mostrar(self):
        return f"({self.x}, {self.y}, {self.z})"


i = VectorEspacio(1, 0, 0)
j = VectorEspacio(0, 1, 0)
k = i.producto_vectorial(j)
print("Dimensión:", VectorEspacio.dimension)
print("i x j =", k.mostrar())

In [ ]:
# Ejercicio 3
class Polinomio:
    variable = "x"

    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self.coeficientes = lista

    def grado(self):
        return len(self.coeficientes) - 1

    def evaluar(self, valor):
        resultado = 0
        for coeficiente in reversed(self.coeficientes):
            resultado = resultado * valor + coeficiente
        return resultado

    def mostrar(self):
        terminos = [
            f"{coeficiente}*{self.variable}^{exponente}"
            for exponente, coeficiente in enumerate(self.coeficientes)
        ]
        return " + ".join(terminos)


polinomio = Polinomio([1, -2, 3, 0, 0])
nulo = Polinomio([0, 0, 0])

print("Coeficientes normalizados:", polinomio.coeficientes)
print("Representación:", polinomio.mostrar())
print("Grado:", polinomio.grado())
print("P(2) =", polinomio.evaluar(2))
print("Polinomio nulo:", nulo.coeficientes, "| grado:", nulo.grado())

## 7. Resumen

- La sentencia `class` construye una plantilla; llamar a la clase crea una instancia nueva e invoca automáticamente a `__init__`.
- El parámetro `self` representa a la instancia concreta que recibe la llamada del método.
- Los **atributos de instancia** viven en `instancia.__dict__` y almacenan el estado propio de cada objeto.
- Los **atributos de clase** viven en `Clase.__dict__` y se comparten entre todas las instancias; conviene reservarlos para valores constantes e inmutables.
- En el siguiente video distinguiremos los tres tipos de métodos dentro de una clase: métodos de instancia, métodos de clase (`@classmethod`) y métodos estáticos (`@staticmethod`).